# Day 1 | ILT 1: Problem Statement & GlobalMart Architecture
### GlobalMart Data Engineering Bootcamp
---
**Duration:** 60 min &nbsp;|&nbsp; **Level:** Beginner &nbsp;|&nbsp; **Tags:** architecture, globalmart, problem-statement

---
**Session Time:** 9:30 AM - 10:30 AM  
**Goal:** Understand WHO GlobalMart is, WHAT their data problem is, and the 2-source Databricks architecture we will build.

---
**INSTRUCTOR NOTE:**  
This is the FIRST session of the entire bootcamp. No code yet — this session is purely about setting context.  
Students need to understand the business problem before they can understand any technical solution.  

By the end of this session, every student should be able to answer:  
- *What does GlobalMart do?*  
- *What is their data problem?*  
- *What are the 2 sources of data we actually use?*  
- *What is `fact_sales` and why does it matter?*  
- *What will we build over 13 days?*

The last 10 minutes have a live code demo where we peek at the actual dataset.

## Learning Objectives

By the end of this session, students will be able to:

1. Describe what GlobalMart is and its data challenges
2. Identify the 2 source systems: Supabase Postgres (via Lakeflow Connect) and ADLS file drops (via Autoloader)
3. Explain why GlobalMart needs a Lakehouse platform on Azure Databricks
4. Name `fact_sales` as the Gold-layer business deliverable and describe its grain
5. Understand the 13-day learning journey ahead

---
## Section 1: Who is GlobalMart?

**INSTRUCTOR NOTE:**  
Start with this. Tell students: *'For the next 13 days, we are Data Engineers at a company called GlobalMart. Everything we build is for this company.'*

---

**GlobalMart** is a mid-sized e-commerce company - think of it as a smaller version of Amazon or Flipkart.

### What does GlobalMart do?

- Sells products across multiple categories (electronics, clothing, home goods, etc.)
- Has customers placing orders online
- Works with multiple suppliers to source products
- Offers multiple payment options (credit card, UPI, cash on delivery)
- Has multiple shipping tiers (standard, express, same-day)
- Handles product returns and refunds

### Key Numbers (GlobalMart scale):

| Metric | Value |
|--------|-------|
| Customers | ~10,000+ |
| Products in catalog | ~100,000+ (products.csv is 100 MB!) |
| Orders per day | Thousands |
| Data sources | 2 main systems (Postgres + ADLS files) |
| Tables we work with | 10 |
| Learning journey | 13 days |

**INSTRUCTOR NOTE:**  
Mention: *'The data you have in your ADLS is real-scale data. products.csv alone is 100 MB. This is why we use Spark and not Excel.'*

---
## Section 2: The Problem Statement

> **Instructor Note:**  
> This section explains **why** the project exists. Read it with energy — the entire bootcamp is focused on solving these business challenges.

---

## GlobalMart's Current Situation

GlobalMart's transactional data lives in **Supabase PostgreSQL**. Product catalogs, customer profiles, and reference files are dropped as CSVs into **Azure Data Lake Storage (ADLS)**. These two systems don't talk to each other automatically.

| Source System | Data Stored | Ingestion Tool |
|--------------|-------------|----------------|
| **Supabase PostgreSQL** | orders, order_items (transactional — continuously updated) | Lakeflow Connect (CDC) |
| **ADLS File Drops** | products, customers, payments, addresses, suppliers, returns, payment_methods, shipping_tier | Autoloader |

> **Day 3 side-exploration only (not part of this pipeline):** REST API (exchange rates) + GraphDB (customer relationships) — useful patterns, not used in the main GlobalMart build.

---

## Business Questions That Are Difficult to Answer Right Now

### Question 1
**Show last month's revenue by product category.**

**Current Challenge**
- Order amounts exist in PostgreSQL
- Product category exists in a CSV file in ADLS
- No automatic join between the two systems

---

### Question 2
**Which products have the highest return rate?**

**Current Challenge**
- Return information is in PostgreSQL
- Product details are in a CSV file
- Analysts manually combine them — different people get different numbers

---

### Question 3
**Which customers placed the most orders last quarter?**

**Current Challenge**
- Orders are in PostgreSQL
- Customer details are in a file drop
- No single trusted view that combines both

---

### Question 4
**What is the payment method breakdown for high-value orders?**

**Current Challenge**
- Order values exist in PostgreSQL
- Payment method lookup is in a CSV file
- Reports take hours to produce manually

---

### Question 5
**Is our data trustworthy?**

**Current Challenge**
- Raw CSV files have nulls, duplicates, and inconsistent formats
- No data quality rules are enforced
- Finance and Operations report different revenue numbers

---

## GlobalMart's Three Major Pain Points

### 1. Data Silos
Transaction data and reference files live in separate systems with no automatic connection.

### 2. No Single Source of Truth
Different teams pull numbers differently — reports don't match.

### 3. Slow Decision Making
Business users wait hours or days for reports that should be instant.

---

## The Solution

GlobalMart needs an **Azure Databricks Lakehouse** that can:

- Ingest orders continuously from PostgreSQL using **Lakeflow Connect (CDC)**
- Ingest reference files automatically from ADLS using **Autoloader**
- Store all raw data in **Bronze** (Delta tables — never modified)
- Clean and standardize data in **Silver** (validated, de-duplicated)
- Build the **`fact_sales` star schema in Gold** — one trusted table for all revenue questions
- Refresh pipelines automatically using **Databricks Workflows**
- Let stakeholders query directly through **Genie** (self-serve Q&A)
- Govern access and lineage through **Unity Catalog**

---

## Target Architecture

```text
Supabase Postgres                         Azure Databricks Lakehouse
(orders, order_items)  ── Lakeflow ──►    Bronze → Silver → Gold (fact_sales)
                            Connect                               ↓
ADLS File Drops        ── Autoloader ──►  Bronze → Silver ──►    Genie
(10 CSV files)                                                    ↓
                                                          DC/Warehouse Manager
```

---

## End Goal

**One Gold Table (`fact_sales`) → One Revenue Number → One Version of the Truth**

Stakeholders ask questions through **Genie** and get trusted answers directly — no waiting for manual reports.

> **Instructor Question:**  
> Have you ever seen Finance report one revenue number while Sales reports a different revenue number?  
>
> That happens because of **Data Silos**.  
>
> The `fact_sales` table we build ensures everyone uses the same trusted source of data.

---
## Section 3: The Two Ingestion Tools

> **Instructor Note:**  
> Spend approximately 15 minutes on this section.  
> The key message: we have **2 real sources**, handled by **2 different Databricks tools**. Everything upstream (Bronze/Silver/Gold) exists to make `fact_sales` correct and current.

---

# Source 1: Supabase (PostgreSQL) via Lakeflow Connect

## What Is It?

Supabase is GlobalMart's primary transactional database.

Every order placed, every payment made, and every item in an order is stored here and updated continuously throughout the day.

**Lakeflow Connect** is Databricks' native CDC (Change Data Capture) connector. It reads changes directly from the Postgres Write-Ahead Log (WAL) and delivers them to the Bronze layer — continuously, without manual scheduling.

### Source Details

| Property | Description |
|-----------|-------------|
| Technology | Supabase (PostgreSQL) |
| Tables ingested | `orders`, `order_items` |
| Data Frequency | Continuous — every insert/update/delete is captured |
| Ingestion Tool | **Lakeflow Connect** (built-in Databricks connector) |
| Method | CDC via Write-Ahead Log (WAL) — no full-table scans |
| Main Challenge | Capturing updates and deletes, not just inserts |
| Bootcamp Coverage | Days 1–5 |

### Understanding WAL (Write-Ahead Log)

Every change to a Postgres table is first written to the WAL before it is committed to the table. Lakeflow Connect reads this log instead of repeatedly querying the full table.

This means:
- INSERT → captured as a new row in Bronze
- UPDATE → captured as an update event
- DELETE → captured as a delete event

No data is missed. No full-table scans on the production database.

---

# Source 2: ADLS File Drops via Autoloader

## What Is It?

Reference data and enrichment files for GlobalMart are deposited as CSV files into an ADLS folder (`raw/`). These files are not live database connections — they are file drops, arriving periodically.

**Autoloader** (`cloudFiles`) is Databricks' incremental file ingestion tool. It monitors a folder in ADLS and automatically picks up new files as they arrive — without re-reading files it has already processed.

### Source Details

| Property | Description |
|-----------|-------------|
| Technology | ADLS Gen2 file drops (CSV files) |
| Tables ingested | `customers`, `products`, `payments`, `addresses`, `suppliers`, `returns`, `payment_methods`, `shipping_tier` |
| Data Frequency | Periodic — files dropped as batches |
| Ingestion Tool | **Autoloader** (`cloudFiles` format) |
| Method | Checkpoint-based — only reads new files, never reprocesses old ones |
| Main Challenge | Schema evolution (new columns appearing in file drops) |
| Bootcamp Coverage | Day 4 hands-on |

### How Autoloader Works

```text
New file lands in raw/        →  Autoloader detects it (via event notification or listing)
                              →  Reads the new file only (checkpoint prevents re-reads)
                              →  Writes to Bronze as Delta
                              →  Updates checkpoint
```

No manual tracking of "which files have I already read" — Autoloader handles this automatically.

---

# Comparing the Two Sources

| Feature | Supabase Postgres (Lakeflow Connect) | ADLS File Drops (Autoloader) |
|---------|--------------------------------------|------------------------------|
| **Data Type** | Live transactional (orders, items) | Batch reference files |
| **Update Pattern** | Continuous CDC | Periodic file drops |
| **Captures** | INSERT + UPDATE + DELETE | New files only |
| **Tool** | Lakeflow Connect | Autoloader (cloudFiles) |
| **Trigger** | Continuous / Scheduled | File arrival |
| **Schema handling** | Fixed schema | Schema evolution supported |
| **Bootcamp days** | Days 1–5 | Day 4 |

---

# The Day 3 Side-Patterns

On Day 3 we also explore two other ingestion patterns that you will encounter in real projects — they do NOT feed the main GlobalMart pipeline:

| Pattern | Tool | Example |
|---------|------|---------|
| REST API | Python `requests` + Autoloader | Exchange rates from frankfurter.app |
| GraphDB | Neo4j Cypher → JDBC → Bronze | Customer-product relationships |

> These are taught as awareness patterns, not as part of the main build. ~80% of the bootcamp is Postgres + ADLS + Delta + Databricks.

---

# Key Takeaway

Two sources. Two tools. One platform.

```text
Supabase Postgres  ──  Lakeflow Connect  ──►
                                             Bronze → Silver → Gold → fact_sales → Genie
ADLS File Drops    ──  Autoloader        ──►
```

By the time data reaches **`fact_sales`**, it has been ingested, cleaned, validated, modelled, and made available to stakeholders — automatically, on schedule, with full lineage.

---
## Section 4: Our Dataset – 10 Tables and Their Sources

> **Instructor Note:**  
> Open the ADLS container and show the files inside the `raw/` folder.  
> Explain which files come from Postgres (via CSV export for the bootcamp) and which are pure file drops.

---

# Dataset Overview

| File | Ingestion Tool | Description |
|--------|---------------|-------------|
| `orders.csv` | Lakeflow Connect (CDC from Postgres) | Order headers — order ID, customer, date, status |
| `order_items.csv` | Lakeflow Connect (CDC from Postgres) | Line items per order — product, quantity, price |
| `customers.csv` | Autoloader (ADLS file drop) | Customer profile information |
| `products.csv` | Autoloader (ADLS file drop) | Product catalog |
| `payments.csv` | Autoloader (ADLS file drop) | Payment transactions |
| `addresses.csv` | Autoloader (ADLS file drop) | Customer delivery addresses |
| `returns.csv` | Autoloader (ADLS file drop) | Returned orders and refund details |
| `suppliers.csv` | Autoloader (ADLS file drop) | Supplier information |
| `payment_methods.csv` | Autoloader (ADLS file drop) | Payment method lookup table |
| `shipping_tier.csv` | Autoloader (ADLS file drop) | Shipping service lookup table |

> **Note:** For this bootcamp, all 10 tables are pre-exported as CSV files in `raw/`. The Lakeflow Connect connection to Supabase is demonstrated live in Day 5 (CDC POC). The pattern is the same — only the delivery mechanism differs.

---

# How the Tables Connect

The dataset follows a relational model where tables are connected through keys.

```text
customers
    |
    +------> orders
                |
                +------> order_items ------> products ------> suppliers
                |
                +------> payments
                |              |
                |              +------> payment_methods
                |
                +------> returns
                |
                +------> shipping_tier

customers
    |
    +------> addresses
```

---

# The Gold Output: fact_sales

All 10 tables ultimately flow into a single star-schema Gold table called **`fact_sales`**.

```text
                     dim_customer
                          ↑
dim_payment_method ← fact_sales → dim_product
                          ↓
                       dim_date
                          ↓
                      dim_address
```

**Grain:** One row per order line item.

**Measures:** `quantity`, `unit_price`, `line_total`

This table answers the question every business team has:
*"What sold, when, to whom, for how much?"*

---

# Why We Need a Lakehouse

Consider the question: *"What was last month's revenue by product category?"*

To answer it you must combine:
```text
orders → order_items → products
```

As datasets grow to millions or billions of records, this join becomes expensive and slow on raw CSV files.

The Lakehouse (Bronze → Silver → Gold) is designed to:
- Centralize data from both sources
- Apply data quality rules in Silver
- Pre-join and pre-aggregate in Gold so `fact_sales` answers the question instantly

---

## Key Takeaway

All 10 tables feed the `fact_sales` star schema in Gold.  
Everything we build in this bootcamp — Bronze ingestion, Silver cleaning, Gold modelling — exists to make that one table correct, current, and trusted.

---
## Section 5: The 13-Day Journey

**INSTRUCTOR NOTE:**  
Show students the full picture of what they will build. This gives them motivation — by the end they will have a complete, production-grade Lakehouse delivering `fact_sales` to Genie.

The arc of the journey: **Bring data in → Clean it → Model it → Refresh it automatically → Lock it down → Serve it to stakeholders.**

---

```
+============================================================================+
|              GLOBALMART 13-DAY LAKEHOUSE JOURNEY                          |
+============================================================================+
|                                                                            |
|  WEEK 1 — Ingestion Foundation (Days 1–5)                                 |
|  ─────────────────────────────────────────                                |
|  Day 1   Problem Statement + Architecture + ADLS + Medallion + Ingestion  |
|  Day 2   Delta Lake deep dive + Code versioning (GitHub) + Idempotency    |
|  Day 3   REST API + GraphDB side patterns (awareness, not main build)     |
|  Day 4   Autoloader hands-on — file drops → Bronze Delta tables           |
|  Day 5   Lakeflow Connect + CDC POC — Postgres WAL → Bronze               |
|          ► Assessment 1 (Ingestion + Delta Lake)                          |
|                                                                            |
|  WEEK 2 — Silver + Modelling (Days 6–10)                                  |
|  ─────────────────────────────────────────                                |
|  Day 6   Bronze → Silver: cleaning, validation, standardisation           |
|  Day 7   SCD Type 1 + Type 2 with MERGE — handling dimension changes      |
|  Day 8   Incremental loading patterns + CDF (Change Data Feed)            |
|  Day 9   Dimensional modelling — star schema design for fact_sales        |
|  Day 10  Build fact_sales: join Bronze/Silver → Gold star schema          |
|                                                                            |
|  WEEK 3 — Orchestration + Governance + Delivery (Days 11–13)             |
|  ─────────────────────────────────────────                                |
|  Day 11  Workflows — automate the full pipeline (scheduled + triggered)   |
|  Day 12  Unity Catalog — access control, lineage, row-level security      |
|  Day 13  Genie — self-serve Q&A on fact_sales for DC/Warehouse Manager   |
|          ► Assessment 2 (End-to-end Lakehouse + Governance)              |
|                                                                            |
+============================================================================+
|  END STATE: Postgres CDC + ADLS files → Bronze → Silver → fact_sales      |
|  refreshed automatically by Workflows, governed by Unity Catalog,          |
|  queryable by stakeholders through Genie — no manual reports needed.      |
+============================================================================+
```

---

### Today's Focus (Day 1):

| Time | Session | What we cover |
|------|---------|---------------|
| 9:30 AM | **ILT 1 (this session)** | Problem statement, architecture, 2 sources, fact_sales |
| 11:00 AM | **ILT 2** | Azure Cloud, ADLS Gen2, Lakehouse vs Warehouse |
| 12:00 PM | **ILT 3** | Medallion Architecture and Delta Lake basics |
| 2:00 PM | **ILT 4** | Intro to Databricks + Lakeflow Connect vs Autoloader |
| 3:30 PM | **Hands-On** | Create ADLS, upload data, connect Databricks, write first Delta tables |

**INSTRUCTOR NOTE:**  
Tell students: *'By the end of today, you will have your own ADLS Gen2 storage set up, all 10 GlobalMart files uploaded, and you will have written your first Delta table. That is a big achievement for Day 1.'*

---
## Section 6: Live Demo - A Quick Look at the GlobalMart Data

**INSTRUCTOR NOTE:**  
Last 10 minutes of this session. Run these code cells to show students what the actual data looks like. This makes the abstract problem statement concrete.  
Keep it quick - we are just peeking at the data, not transforming anything yet.

Make sure your cluster is running before you start.

In [ ]:
# ============================================================
# CELL 1: Connect to ADLS and define paths
#
# INSTRUCTOR NOTE:
#   Same setup as ILT 2. Paste your storage key and run.
# ============================================================

storage_account_name = "amazonprojectadls"
container_name       = "amazon-data"
raw_folder           = "raw"
storage_account_key  = "YOUR_STORAGE_ACCOUNT_KEY"   # paste your key here

spark.conf.set(
    f"fs.azure.account.key.{storage_account_name}.dfs.core.windows.net",
    storage_account_key
)

raw_path = f"abfss://{container_name}@{storage_account_name}.dfs.core.windows.net/{raw_folder}"

print("Connected to ADLS.")
print(f"Reading from: {raw_path}")

In [ ]:
# ============================================================
# CELL 2: Get a row count for all 10 tables
#
# INSTRUCTOR NOTE:
#   Show students the scale of data we are working with.
#   Notice products.csv has the most rows - it is the full product catalog.
#   orders_items has more rows than orders because each order has multiple products.
# ============================================================

tables = [
    "customers", "orders", "orders_items", "products",
    "payments", "addresses", "returns", "suppliers",
    "payment_methods", "shipping_tier"
]

print("GlobalMart Dataset Overview")
print("=" * 45)
print(f"{'Table':<25} {'Rows':>10}  Source")
print("-" * 55)

source_map = {
    "customers": "OLTP DB", "orders": "OLTP DB",
    "orders_items": "OLTP DB", "payments": "OLTP DB",
    "addresses": "OLTP DB", "returns": "OLTP DB",
    "products": "Flat File", "suppliers": "Flat File",
    "payment_methods": "Flat File", "shipping_tier": "Flat File"
}

for table in tables:
    df = spark.read.option("header", "true").csv(f"{raw_path}/{table}.csv")
    count = df.count()
    print(f"{table:<25} {count:>10,}  {source_map[table]}")

print("-" * 55)

In [ ]:
# ============================================================
# CELL 3: Peek at customers - who are they?
#
# INSTRUCTOR NOTE:
#   Show this to make the data human. These are the people
#   buying from GlobalMart. Point out columns - what is useful?
#   What might be messy? (nulls, email formats, etc.)
#   We will fix these issues in Week 2 (Silver transformation).
# ============================================================

df_customers = spark.read.option("header", "true").csv(f"{raw_path}/customers.csv")

print("Sample customers:")
df_customers.show(5, truncate=True)

print("Columns:", df_customers.columns)

In [ ]:
# ============================================================
# CELL 4: Peek at orders - what does a transaction look like?
#
# INSTRUCTOR NOTE:
#   Orders is the central table - most analysis will start here.
#   Notice the 'status' column - orders can be pending, shipped,
#   delivered, cancelled. This is a key field for business analytics.
# ============================================================

df_orders = spark.read.option("header", "true").csv(f"{raw_path}/orders.csv")

print("Sample orders:")
df_orders.show(5, truncate=True)

print("Columns:", df_orders.columns)

In [ ]:
# CELL 5: Show the business problem - data is in silos right now
#
# INSTRUCTOR NOTE:
#   This cell demonstrates WHY we need a Lakehouse.
#   The manager's question: 'How many orders does each customer have?'
#   Right now we CAN answer this because we have both tables.
#   But imagine if customers were in one database and orders in another
#   system with no connection - this join would be impossible without
#   a unified platform.
#
#   This is a preview of what Silver and Gold will look like.
# ============================================================

# A simple business question: which customers placed the most orders?
# ============================================================
# CELL 5: Show the business problem - data is in silos right now
# ============================================================

print("Business Question: Which customers placed the most orders?")
print("(This requires joining customers + orders - two separate source tables)")
print()

# Read source tables
df_customers = spark.read.option("header", "true").csv(f"{raw_path}/customers.csv")
df_orders = spark.read.option("header", "true").csv(f"{raw_path}/orders.csv")

from pyspark.sql.functions import count, col

# Count orders per customer
df_order_counts = (
    df_orders
    .groupBy("CustomerID")
    .agg(count("OrderID").alias("total_orders"))
)

# Join with customer information
df_result = (
    df_order_counts
    .join(df_customers, on="CustomerID", how="left")
    .select(
        "CustomerID",
        "FirstName",
        "LastName",
        "total_orders"
    )
    .orderBy(col("total_orders").desc())
)

print("Top 10 customers by order count:")

display(df_result.limit(10))

# print()
# print("This is what Gold layer will look like - pre-joined, aggregated, business-ready.")

# print()
# print("This is what Gold layer will look like - pre-joined, aggregated, business-ready.")

---
## Session Summary

| Topic | Key Takeaway |
|-------|-------------|
| **GlobalMart** | E-commerce company — transactional data in Postgres, reference files in ADLS |
| **The problem** | Data silos → no single truth → slow decisions → teams don't trust reports |
| **Source 1** | Supabase Postgres → ingested via **Lakeflow Connect** (CDC, continuous) |
| **Source 2** | ADLS file drops → ingested via **Autoloader** (incremental, checkpoint-based) |
| **Bronze** | Raw data exactly as received — never modified |
| **Silver** | Cleaned, validated, deduplicated — row-level Delta tables |
| **Gold** | `fact_sales` star schema — one trusted table answering all revenue questions |
| **Genie** | Self-serve Q&A on `fact_sales` — replaces manual email reports |
| **13-day goal** | Fully automated, governed Lakehouse serving `fact_sales` through Genie |

---

### Day 3 side-exploration (not the main pipeline):
REST API (exchange rates) + GraphDB (customer relationships) — awareness patterns only.

---

### What Is Coming Next:

| Session | Topic |
|---------|-------|
| **ILT 2 (11:00 AM)** | Azure Cloud, ADLS Gen2, Lakehouse vs Warehouse vs Hybrid |
| **ILT 3 (12:00 PM)** | Medallion Architecture and Delta Lake Fundamentals |
| **ILT 4 (2:00 PM)** | Intro to Databricks + Lakeflow Connect vs Autoloader |
| **Hands-On (3:30 PM)** | Create ADLS, upload data, connect Databricks, write first Delta table |

---

**INSTRUCTOR NOTE:**  
Closing activity (5 minutes): Ask each student to answer one question out loud:
1. *'Name the 2 data sources GlobalMart uses in this pipeline.'* (Postgres + ADLS files)
2. *'What is the name of the Gold table we build?'* (`fact_sales`)
3. *'What are the 3 layers of the Medallion Architecture?'* (Bronze, Silver, Gold)